# Practical - 03 : AWS Bedrock Hello World

> **Prerequisites**
> - AWS Account
> - IAM User with Bedrock permissions
> - AWS CLI configured
> - Python 3.11+
> - boto3

In [1]:
"""Importing bot3 library for the project"""
import boto3
print(boto3.__version__)

"""Verify that boto3 (AWS SDK for Python) is installed correctly.

Why?
Every interaction with Amazon Bedrock in this project is performed
through boto3. Before making AWS API calls, we ensure that the SDK is
available and inspect the installed version.
"""

1.36.3


'Verify that boto3 (AWS SDK for Python) is installed correctly.\n\nWhy?\nEvery interaction with Amazon Bedrock in this project is performed\nthrough boto3. Before making AWS API calls, we ensure that the SDK is\navailable and inspect the installed version.\n'

#### Checking out the credentials systems

In [2]:
import boto3

session = boto3.Session()

credentials = session.get_credentials()

print("Credentials:", credentials)
print("Region:", session.region_name)

Credentials: <botocore.credentials.Credentials object at 0x108fd4830>
Region: us-east-1


In [3]:
"""Make the project root importable, so utils, tests, src works properly"""
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.config import config
print("Region:", config.region_name)
print("Default model:", config.default_model_id)

# Loads the project configurations

Region: us-east-1
Default model: amazon.nova-micro-v1:0


## List all available Bedrock foundation models

In [4]:
# # Invokes control-plane client ListFoundationModels to retrieve available model metadata without performing any model inference.
from src.list_bedrock_models import list_foundation_models, print_models_table

all_models = list_foundation_models()
print(f"Found {len(all_models)} foundation models available in {config.region_name}.")
print_models_table(all_models)

Region: us-east-1
Credentials: <botocore.credentials.Credentials object at 0x109014550>
Available profiles: ['default']
Found 121 foundation models available in us-east-1.
MODEL ID                                      PROVIDER     MODEL NAME               
------------------------------------------------------------------------------------
nvidia.nemotron-nano-12b-v2                   NVIDIA       NVIDIA Nemotron Nano 12B v2 VL BF16
qwen.qwen3-coder-next                         Qwen         Qwen3 Coder Next         
anthropic.claude-sonnet-4-20250514-v1:0       Anthropic    Claude Sonnet 4          
anthropic.claude-haiku-4-5-20251001-v1:0      Anthropic    Claude Haiku 4.5         
moonshotai.kimi-k2.5                          Moonshot AI  Kimi K2.5                
openai.gpt-oss-120b-1:0                       OpenAI       gpt-oss-120b             
stability.stable-creative-upscale-v1:0        Stability AI Stable Image Creative Upscale
qwen.qwen3-next-80b-a3b                       Qwe

In [5]:
""" Retrieve only Amazon foundation models and display the Amazon Nova family used throughout this practical."""
""" Why? --- Amazon Nova models fully support the Converse API and do not require the additional provider access request needed by some third-party models."""

# Retrieve all Amazon foundation models
amazon_models = list_foundation_models(provider_filter="amazon")

# Keep only Amazon Nova models for this practical
nova_models = [
    model
    for model in amazon_models
    if "nova" in model["modelId"].lower()
]

print("--- Amazon Nova Models ---")
print_models_table(nova_models)

--- Amazon Nova Models ---
MODEL ID                                      PROVIDER     MODEL NAME               
------------------------------------------------------------------------------------
amazon.nova-2-multimodal-embeddings-v1:0      Amazon       Amazon Nova Multimodal Embeddings
amazon.nova-pro-v1:0                          Amazon       Nova Pro                 
amazon.nova-pro-v1:0:24k                      Amazon       Nova Pro                 
amazon.nova-pro-v1:0:300k                     Amazon       Nova Pro                 
amazon.nova-2-lite-v1:0                       Amazon       Nova 2 Lite              
amazon.nova-2-lite-v1:0:256k                  Amazon       Nova 2 Lite              
amazon.nova-2-sonic-v1:0                      Amazon       Nova 2 Sonic             
amazon.nova-premier-v1:0:8k                   Amazon       Nova Premier             
amazon.nova-premier-v1:0:20k                  Amazon       Nova Premier             
amazon.nova-premier-v1:0:1000k

# Invoke a model via the Converse API

#### Invokes data-plane bedrock-runtime Converse API for model-agnostic chat, printing the model response, stop reason, and token usage statistics.


In [6]:
from src.converse import send_prompt

result = send_prompt(
    prompt="In two sentences, explain what Amazon Bedrock is to a backend engineer.",
    model_id=config.default_model_id,
)

print("MODEL RESPONSE:\n", result["text"])
print("\nStop reason:", result["stop_reason"])
print(f"Tokens — input: {result['input_tokens']}, output: {result['output_tokens']}")

MODEL RESPONSE:
 Amazon Bedrock is a robust, cloud-based infrastructure service designed to simplify the deployment and management of machine learning models, offering scalable and secure backend support for developers. It abstracts complex infrastructure details, allowing backend engineers to focus on model development rather than operational overhead.

Stop reason: end_turn
Tokens — input: 15, output: 53


### Same prompt, different model (NovaMicro vs NovaLite)

Because `send_prompt()` takes `model_id` as a parameter (not hardcoded),
switching models is a one-line change — this is the practical benefit of
using the Converse API instead of `invoke_model()`, which would need a
completely different request-building function per provider.

In [7]:
prompt = "In two sentences, explain what Amazon Bedrock is to a backend engineer."

nova_result = send_prompt(prompt=prompt, model_id="amazon.nova-micro-v1:0")
nova_result = send_prompt(prompt=prompt, model_id="amazon.nova-lite-v1:0")


print(nova_result["text"])
print(nova_result["text"])

Amazon Bedrock is a fully managed service designed to help developers build, share, and scale generative AI applications by providing a foundation of pre-trained models and tools for customization. It simplifies the integration of advanced AI capabilities into backend systems, allowing engineers to focus on developing innovative features rather than managing underlying infrastructure.
Amazon Bedrock is a fully managed service designed to help developers build, share, and scale generative AI applications by providing a foundation of pre-trained models and tools for customization. It simplifies the integration of advanced AI capabilities into backend systems, allowing engineers to focus on developing innovative features rather than managing underlying infrastructure.


In [8]:
from src.converse import send_prompt

prompt = "Who is the best footballer, Cristiano Ronaldo or Lionel Messi? Compare them based on achievements, playing style, statistics, and overall impact on football. Finally, give a balanced conclusion."

result = send_prompt(
    prompt=prompt,
    model_id="amazon.nova-micro-v1:0"
)

print(result["text"])

Comparing Cristiano Ronaldo and Lionel Messi is a subjective exercise that often sparks debate among football fans. Both players have achieved extraordinary success and left an indelible mark on the sport. Here's a detailed comparison based on various aspects:

### Achievements

**Cristiano Ronaldo:**
- **FIFA World Player of the Year:** 5 times (2007, 2008, 2013, 2014, 2016)
- **Ballon d'Or:** 5 times (2008, 2013, 2014, 2016, 2017)
- **UEFA Champions League Titles:** 5 (2008, 2014, 2016, 2017, 2018)
- **Premier League Titles:** 3 (2008, 2011, 2017)
- **La Liga Titles:** 3 (2008, 2012, 2013)
- **Copa del Rey:** 2 (2013, 2014)
- **UEFA Euro:** 1 (2016)
- **UEFA Nations League:** 1 (2019)

**Lionel Messi:**
- **FIFA World Player of the Year:** 6 times (2009, 2010, 2011, 2012, 2015, 2019)
- **Ballon d'Or:** 7 times (2009, 2010, 2011, 2012, 2015, 2019, 2021)
- **UEFA Champions League Titles:** 4 (2006, 2009, 2011, 2015)
- **La Liga Titles:** 10 (2009, 2010, 2013, 2018, 2019, 2020, 2021)
- 

In [9]:
from src.converse import send_prompt

prompt = "Why is Rohit Sharma often compared to MS Dhoni in terms of captaincy? Compare their leadership styles, ICC achievements, IPL success, decision-making, handling pressure, and impact on Indian cricket. End with a balanced conclusion."

result = send_prompt(
    prompt=prompt,
    model_id="amazon.nova-micro-v1:0"
)

print(result["text"])

Rohit Sharma and MS Dhoni are two of the most iconic figures in Indian cricket, and their leadership styles, particularly as captains, have often been compared due to their significant impacts on the team's performance and success. Here’s a detailed comparison across various aspects:

### Leadership Styles

**MS Dhoni:**
- **Calm and Composed:** Dhoni is known for his calm demeanor, often referred to as the "coolest" captain in cricket. His composed nature under pressure is legendary.
- **Strategic and Tactical:** He has an excellent understanding of the game and often employs unconventional strategies, such as promoting spinners early in ODIs.
- **Mentorship:** Dhoni has been a mentor to many young players, guiding them through their cricketing careers.

**Rohit Sharma:**
- **Enthusiastic and Energetic:** Rohit brings a more enthusiastic and energetic approach to his captaincy. His leadership is characterized by a positive and motivating attitude.
- **Inclusive and Team-Oriented:** He

## 3. Experimenting with Temperature and Top_p

In this section, we explore how different generation parameters affect the model's response.

### Temperature
`temperature` controls the creativity of the model's output.

- **Low (0.1 – 0.3):** More focused, predictable, and consistent responses.
- **Medium (0.5 – 0.7):** Balanced between accuracy and creativity.
- **High (0.8 – 1.0):** More diverse and creative responses, but they may become less consistent.

**Best used for:**
- Low → Code generation, Q&A, summarization
- High → Brainstorming, storytelling, content creation

---

### Top-p (Nucleus Sampling)
`top_p` controls how many possible next words the model considers before generating the response.

- **Low (0.3 – 0.5):** Chooses from only the most likely words, producing safer and more focused responses.
- **High (0.9 – 1.0):** Considers a wider range of words, resulting in more varied and natural responses.

---

### Why use both?

Although both parameters influence randomness, they work differently:

- **Temperature** changes *how creative* the model becomes.
- **Top-p** changes *how many word choices* the model can select from.

By experimenting with different values, we can observe how the style, creativity, and consistency of the generated text change.

In [10]:
creative_prompt = "Write a one-sentence tagline for a startup that helps engineers learn AI faster."

experiment_settings = [
    {"label": "Low temperature (0.1), high top_p (0.9)", "temperature": 0.1, "top_p": 0.9},
    {"label": "High temperature (1.0), high top_p (0.9)", "temperature": 1.0, "top_p": 0.9},
    {"label": "High temperature (1.0), low top_p (0.3)",  "temperature": 1.0, "top_p": 0.3},
]

for setting in experiment_settings:
    result = send_prompt(
        prompt=creative_prompt,
        temperature=setting["temperature"],
        top_p=setting["top_p"],
    )
    print(f"[{setting['label']}]")
    print(result["text"])
    print("-" * 70)


[Low temperature (0.1), high top_p (0.9)]
"Accelerate your AI mastery—learn faster, engineer smarter with our cutting-edge training platform."
----------------------------------------------------------------------
[High temperature (1.0), high top_p (0.9)]
"Accelerate your AI mastery—learn faster, innovate smarter with our engineering-focused platform."
----------------------------------------------------------------------
[High temperature (1.0), low top_p (0.3)]
"Accelerate your AI mastery—learn faster, engineer smarter with our cutting-edge training platform."
----------------------------------------------------------------------


## Run the same setting multiple times

A single run at temperature=1.0 doesn't prove anything by itself — the
interesting comparison is **variance across repeated calls** at the same
setting. Low temperature should give near-identical outputs each time;
high temperature should give visibly different outputs each time.

In [11]:
print("=== 3 runs at temperature=0.1 (expect near-identical outputs) ===")
for i in range(3):
    r = send_prompt(prompt=creative_prompt, temperature=0.1, top_p=0.9)
    print(f"Run {i+1}: {r['text']}")

print("\n=== 3 runs at temperature=1.0 (expect noticeably different outputs) ===")
for i in range(3):
    r = send_prompt(prompt=creative_prompt, temperature=1.0, top_p=0.9)
    print(f"Run {i+1}: {r['text']}")

=== 3 runs at temperature=0.1 (expect near-identical outputs) ===
Run 1: "Accelerate your AI mastery—learn faster, engineer smarter with our cutting-edge training platform."
Run 2: "Accelerate your AI mastery—learn faster, engineer smarter with our cutting-edge training platform."
Run 3: "Accelerate your AI journey: Learn faster, innovate smarter with our engineer-focused AI training."

=== 3 runs at temperature=1.0 (expect noticeably different outputs) ===
Run 1: "Accelerate your AI expertise—learn faster, code smarter with our innovative engineer-focused platform."
Run 2: "Accelerate your engineering journey: Master AI faster with our tailored learning platform."
Run 3: "Accelerate your AI expertise faster with our tailored learning platform for engineers."
